# Knee MRI: 5-fold retrain on calibrated v4 labels (`v4mix`)
Trains 5 EfficientNet-B0 fold models (160 px, baseline recipe) on `labels_soft_v4.csv`, scored on the FIXED old labels (`--eval_labels oof_all.csv`) with the same site folds as the baseline.
GPU 0 runs folds 0-2, GPU 1 runs folds 3-4 (~3.3 h wall; GPU 1 idles for the last ~1 h). Then it merges the two `oof.csv`, renames the checkpoints `fold{k}_v4mix.pt` (so they can sit next to the old `fold{k}.pt` and still match `infer.py`'s `fold*.pt` glob) and compares with the baseline over ALL folds.

**Settings:** GPU T4 x2, internet ON, *Save & Run All*. **Inputs:** `knee-mri-pipeline` (v5 code), label notebook output (`labels_soft_v4.csv`), caching notebook output (160 px cache), baseline training output (`oof_all.csv`). The competition data is not needed.

In [1]:
import os, sys, glob, json, shutil, subprocess, time
import numpy as np, pandas as pd

LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
          "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
KEY4 = LABELS[:4]                      # ACL, MCL, Medial Meniscus, Lateral Meniscus
W = "/kaggle/working"

def launch(gpu, out, extra, code, train_args):
    """Start one train.py process pinned to one GPU. Returns (Popen, logfile) or None if already running."""
    name = os.path.basename(out)
    log = f"{W}/{name}.log"
    if subprocess.run(["pgrep", "-f", "--", f"--out {out}"], capture_output=True).returncode == 0:
        print(f"[{name}] already running -> skipped (launch guard)")
        return None
    cmd = [sys.executable, f"{code}/train.py", *train_args, "--out", out, *extra]
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    print(f"[{name}] GPU {gpu}:", " ".join(cmd))
    return subprocess.Popen(cmd, env=env, stdout=open(log, "w"), stderr=subprocess.STDOUT), log

def last_match(log, pat):
    try:
        lines = [l.rstrip() for l in open(log, errors="ignore") if pat in l]
        return lines[-1] if lines else "(none yet)"
    except FileNotFoundError:
        return "(no log)"

def wait_all(jobs, every=900):
    """Block until every process ends (REQUIRED in a committed notebook: it kills children when the last cell ends)."""
    t0, last = time.time(), 0.0
    while any(p.poll() is None for p, _ in jobs.values()):
        time.sleep(20)
        if time.time() - last >= every:
            last = time.time()
            print(f"[{(time.time() - t0) / 3600:.2f} h]", {k: last_match(l, " ep ") for k, (_, l) in jobs.items()}, flush=True)
    bad = []
    for k, (p, l) in jobs.items():
        print(f"{k} exit code {p.returncode}")
        if p.returncode != 0:
            bad.append(k); print("---- tail of", l); print("".join(open(l, errors="ignore").readlines()[-25:]))
    return bad

CODE  = "/kaggle/input/datasets/theokorir/knee-mri-pipeline/knee_pipeline"
CACHE = "/kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/cache"
BASE  = "/kaggle/input/notebooks/theokorir/knee-mri-diagnosis-training/oof_all.csv"
LAB   = "/kaggle/input/notebooks/theokorir/knee-mri-diagnosis-label"

for p in (CODE, CACHE, BASE, LAB):
    assert os.path.exists(p), f"missing input: {p}"
src = open(f"{CODE}/train.py").read()
assert "--eval_labels" in src, "dataset holds an OLD train.py (no --eval_labels). Upload the newest knee_pipeline_v5.zip."
c = glob.glob(f"{LAB}/**/labels_soft_v4.csv", recursive=True)
assert len(c) == 1, f"labels_soft_v4.csv: {len(c)} found: {c}"
LABELS_V4 = c[0]
lab, base = pd.read_csv(LABELS_V4), pd.read_csv(BASE)
need = ["StudyInstanceUID", "is_gold"] + [f"{p}_{l}" for p in "yw" for l in LABELS]
assert not [x for x in need if x not in lab.columns], [x for x in need if x not in lab.columns]
print(f"v4 labels: {LABELS_V4} | rows {len(lab)} | gold {int(lab.is_gold.sum())}")
acl = lab["y_ACL"].mean()
print(f"mean y_ACL {acl:.3f}  (the fold-0 v4mix run on 5 Oct showed 0.155)")
assert len(lab) == 4407 and abs(acl - 0.155) < 0.01, "this is not the labels_soft_v4.csv used for the fold-0 v4mix run: STOP"
assert "fold" in base.columns, "oof_all.csv has no fold column: fold equality with the baseline could not be enforced by train.py"
print("setup OK")

v4 labels: /kaggle/input/notebooks/theokorir/knee-mri-diagnosis-label/labels_soft_v4.csv | rows 4407 | gold 58
mean y_ACL 0.155  (the fold-0 v4mix run on 5 Oct showed 0.155)
setup OK


In [2]:
TRAIN_ARGS = ["--cache_dir", CACHE, "--labels", LABELS_V4, "--eval_labels", BASE, "--epochs", "12", "--pretrained",
              "--workers", "2", "--train_slices", "16", "--bs", "4", "--accum", "2"]
jobs = {}
for gpu, folds, name in ((0, ["0", "1", "2"], "v4mix_g0"), (1, ["3", "4"], "v4mix_g1")):
    r = launch(gpu, f"{W}/{name}", ["--only_folds", *folds], CODE, TRAIN_ARGS)
    if r: jobs[name] = r
time.sleep(150)
for k, (p, l) in jobs.items():
    print(k, "|", last_match(l, "studies:"), "|", last_match(l, "===== fold"))
bad = wait_all(jobs)
assert not bad, f"failed: {bad}"

[v4mix_g0] GPU 0: /usr/bin/python3 /kaggle/input/datasets/theokorir/knee-mri-pipeline/knee_pipeline/train.py --cache_dir /kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/cache --labels /kaggle/input/notebooks/theokorir/knee-mri-diagnosis-label/labels_soft_v4.csv --eval_labels /kaggle/input/notebooks/theokorir/knee-mri-diagnosis-training/oof_all.csv --epochs 12 --pretrained --workers 2 --train_slices 16 --bs 4 --accum 2 --out /kaggle/working/v4mix_g0 --only_folds 0 1 2
[v4mix_g1] GPU 1: /usr/bin/python3 /kaggle/input/datasets/theokorir/knee-mri-pipeline/knee_pipeline/train.py --cache_dir /kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/cache --labels /kaggle/input/notebooks/theokorir/knee-mri-diagnosis-label/labels_soft_v4.csv --eval_labels /kaggle/input/notebooks/theokorir/knee-mri-diagnosis-training/oof_all.csv --epochs 12 --pretrained --workers 2 --train_slices 16 --bs 4 --accum 2 --out /kaggle/working/v4mix_g1 --only_folds 3 4
v4mix_g0 | studies: 4407 | gold:

In [3]:
import torch
os.makedirs(f"{W}/models_v4mix", exist_ok=True)
for g, folds in (("v4mix_g0", (0, 1, 2)), ("v4mix_g1", (3, 4))):
    for k in folds:
        src_pt = f"{W}/{g}/fold{k}.pt"
        assert os.path.exists(src_pt), f"missing {src_pt}"
        ck = torch.load(src_pt, map_location="cpu")
        assert {"state", "backbone", "K"} <= set(ck), list(ck)
        shutil.copy(src_pt, f"{W}/models_v4mix/fold{k}_v4mix.pt")
print(sorted(os.listdir(f"{W}/models_v4mix")))
o0, o1 = (pd.read_csv(f"{W}/{g}/oof.csv") for g in ("v4mix_g0", "v4mix_g1"))
pc = [f"p_{l}" for l in LABELS]
m1 = o1.p_ACL.notna()
assert (o0.StudyInstanceUID == o1.StudyInstanceUID).all() and not (o0.p_ACL.notna() & m1).any(), "the two processes overlap or are misaligned"
merged = o0.copy(); merged.loc[m1, pc] = o1.loc[m1, pc].values
assert merged.p_ACL.notna().all(), f"{int(merged.p_ACL.isna().sum())} studies unscored"
merged.to_csv(f"{W}/oof_v4mix.csv", index=False)
print("saved oof_v4mix.csv", merged.shape)

['fold0_v4mix.pt', 'fold1_v4mix.pt', 'fold2_v4mix.pt', 'fold3_v4mix.pt', 'fold4_v4mix.pt']
saved oof_v4mix.csv (4407, 40)


In [4]:
from sklearn.metrics import roc_auc_score

def aucs(Y, P, rows=None):
    """Per-label AUC on CONFIDENT old labels (y>=0.9 positive, y<=0.1 negative). Y, P: [N,12]."""
    out = []
    for j in range(12):
        y = Y[:, j]; m = (y >= 0.9) | (y <= 0.1)
        if rows is not None: m = m & rows
        t = (y[m] >= 0.9).astype(int)
        out.append(roc_auc_score(t, P[m, j]) if 0 < t.sum() < len(t) else np.nan)
    return np.array(out)

summ = lambda a: (float(np.nanmean(a)), float(np.nanmean(a[:4])))     # (macro, KEY-4)

from scipy.stats import rankdata
new = pd.read_csv(f"{W}/oof_v4mix.csv").set_index("StudyInstanceUID")
b = base.set_index("StudyInstanceUID").loc[new.index]
pc, yc = [f"p_{l}" for l in LABELS], [f"y_{l}" for l in LABELS]
Y, Pb, Pn, fold = b[yc].values, b[pc].values, new[pc].values, new.fold.values
assert (b.fold.values == fold).all(), "folds differ from the baseline"
print("max |y_ new - y_ baseline|:", np.nanmax(np.abs(new[yc].values - Y)))

def frank(P):
    R = np.zeros_like(P, dtype=float)
    for k in np.unique(fold):
        m = fold == k
        for j in range(12): R[m, j] = rankdata(P[m, j]) / m.sum()
    return R
Rb, Rn = frank(Pb), frank(Pn); Re = (Rb + Rn) / 2     # per-fold ranks, as infer.py rank-averages

T = pd.DataFrame({"baseline": aucs(Y, Rb), "v4mix": aucs(Y, Rn), "ensemble (rank mean)": aucs(Y, Re)}, index=LABELS)
print("\nAll 5 folds, pooled with per-fold ranks, confident old labels:"); print(T.round(3).to_string())
S = {c: summ(T[c].values) for c in T.columns}
print("\n                        macro   KEY-4"); [print(f"{c:22s} {v[0]:.4f}  {v[1]:.4f}") for c, v in S.items()]

print("\nper-fold macro (raw predictions):")
for k in np.unique(fold):
    m = fold == k
    print(f"  fold {k}: baseline {summ(aucs(Y, Pb, m))[0]:.4f} | v4mix {summ(aucs(Y, Pn, m))[0]:.4f}")

gold = new.is_gold.values.astype(bool)
print(f"\ngold ({gold.sum()} studies; 95% CIs about +/-0.1, NOT decisive):")
for nme, R in (("baseline", Rb), ("v4mix", Rn), ("ensemble", Re)):
    print(f"  {nme:9s} macro {summ(aucs(Y, R, gold))[0]:.4f}")

rng, n = np.random.default_rng(0), len(Y)
for nme, R in (("v4mix - baseline", Rn), ("ensemble - baseline", Re)):
    d = []
    for _ in range(300):
        i = rng.integers(0, n, n); d.append(summ(aucs(Y[i], R[i]))[1] - summ(aucs(Y[i], Rb[i]))[1])
    print(f"KEY-4 {nme}: {S['v4mix' if nme.startswith('v4') else 'ensemble (rank mean)'][1] - S['baseline'][1]:+.4f}  90% CI [{np.percentile(d, 5):+.4f}, {np.percentile(d, 95):+.4f}]")
print("\nNOTE: this OOF ensemble is 1 old + 1 new model per study. At test time each side is already a 5-model average,"
      "\nso the real gain could differ. Seed noise on one fold was ~0.005 KEY-4. Judge by the public LB vs 0.829 only as a sanity check.")

max |y_ new - y_ baseline|: 0.0

All 5 folds, pooled with per-fold ranks, confident old labels:
                  baseline  v4mix  ensemble (rank mean)
ACL                  0.781  0.804                 0.798
MCL                  0.725  0.716                 0.723
Medial Meniscus      0.770  0.772                 0.774
Lateral Meniscus     0.736  0.734                 0.738
Medial OA            0.826  0.826                 0.829
Lateral OA           0.747  0.740                 0.746
PF OA                0.717  0.717                 0.720
Effusion             0.822  0.808                 0.818
Synovitis            0.772  0.766                 0.771
Baker's              0.756  0.734                 0.750
Contusion            0.755  0.747                 0.754
Fracture             0.780  0.777                 0.782

                        macro   KEY-4
baseline               0.7657  0.7531
v4mix                  0.7619  0.7567
ensemble (rank mean)   0.7667  0.7582

per-fold macro (raw pr